# 世界モデルと深層生成モデル

世界モデルと深層生成モデルでは、未来予測を使った意思決定を、実装可能な形に分解して学びます。


## 背景と目的

世界モデルは生成モデルを予測と制御に接続するための枠組みです。

両者の関係を整理すると、表現学習を行動計画へ結びつけられます。

生成と遷移予測の役割分担を確認します。


## 最初に解きたい疑問

1. 「世界モデル」と「深層生成モデル」は、どこが同じでどこが違うのか。
2. `z_t` は何を表していて、なぜ観測 `o_t` そのものを使わないのか。
3. `A` と `B` の値は何を意味し、学習ではどう決まるのか。
4. `rollout` で複数ステップ回すと、何を確認できるのか。
5. `score_plan` のように計画候補を比べるとき、何を基準に良し悪しを決めるのか。


## 先に押さえる言葉

- `潜在状態 z`: 観測をそのまま使わず、モデルの中で要約して持つ内部の状態です。次の予測や計画に使うための圧縮表現です。
- `遷移モデル f_theta`: 今の状態と行動から、次の状態を予測する部分です。世界がどう変わるかのルールを学習したものとして扱います。
- `観測復元 g_theta`: 潜在状態から、見える形の観測を戻す部分です。内部表現が外部の観測に対応しているかを確認します。
- `ロールアウト`: 1回の予測ではなく、予測を何ステップも連続して進めることです。誤差が積み上がる様子を見るために使います。


## 実行前の見取り図

1. `z_next` の値が行動 `a_t` を変えると素直に変わるか。
2. `rollout = [...]` が途中で急に発散せず、各ステップの値を追えるか。
3. `errors` と `mean_error` が出て、どの時点で誤差が大きいか読めるか。


## 出力の読み方

- `rollout` が途中でどう劣化するかの読み方。
- `errors` と `mean_error` のどちらを見るべきか。


## つまずきやすい点

- `z_t` と `obs` の役割分担を、図なしでも分かるようにもう一段だけ言い換える説明。
- `score_plan` が何を最大化または最小化しているのかを補足する説明。
- z_t と obs の役割分担の言い換え。
- score_plan が何を最適化しているかの補足。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- ここでの世界モデルは概念実験であり、本格的な学習済みモデルではない点。


## 観察 1: 世界モデルの遷移初期化

生成モデルとの関係を見る前に、遷移式の基礎を定義します。


In [ ]:
z_t = 0.20
a_t = 1.0
A, B = 0.90, 0.20
z_next = A * z_t + B * a_t
print('task = world-models-and-generative-models')
print('z_next =', round(z_next, 6))

この遷移を起点に観測復元へ進みます。



## 観察 2: 観測予測を作る

次に、潜在状態から観測を復元する写像を作ります。状態推定と観測再現の役割分担をコードで掴みます。


In [ ]:
def decode(z):
    return {'position': 2.5 * z + 0.1, 'velocity': 0.8 * z - 0.05}
obs_next = decode(z_next)
print('obs_next =', {k: round(v, 4) for k, v in obs_next.items()})
print('keys =', list(obs_next.keys()))

観測予測を別関数に切ると、遷移誤差と観測誤差を分離して調整できます。



## 計算の対応表

1. $z_{t+1} = f_{\theta}(z_t, a_t)$
2. $\hat{o}_{t+1} = g_{\theta}(z_{t+1})$


## 観察 3: ロールアウトを試す

ここで複数ステップ予測を実行します。1ステップでは見えない誤差累積を把握するためです。


In [ ]:
actions = [0.0, 1.0, 1.0, 0.0, -0.5]
z = 0.1
traj = []
for a in actions:
    z = 0.92 * z + 0.18 * a
    traj.append(round(z, 5))
print('rollout =', traj)

長期予測で崩れるなら、遷移モデルの安定性や状態表現の情報量不足を疑います。



## 観察 4: 計画候補を比較する

次に、複数の行動列を比較して、どの計画が望ましいかを評価します。モデルベース強化学習の中心操作です。


In [ ]:
plans = [[0, 1, 1], [1, 1, 1], [0, 0, 1]]
def score_plan(plan):
    z = 0.1
    for a in plan:
        z = 0.92 * z + 0.18 * a
    return z
scores = [round(score_plan(p), 5) for p in plans]
print('scores =', scores)

この `score_plan` は、最終潜在値が高いほど良いと仮定した教育用の代理スコアです。実際の計画では、予測した状態から報酬・コスト・制約違反を計算して評価します。ここで使う `A` と `B` も、本番ではデータから学習される遷移係数です。


計画評価が可能になると、実環境での試行回数を抑えた探索がしやすくなります。



## 観察 5: モデル誤差を監視する

最後に、予測と実測の差を定量化します。世界モデルは『予測できる範囲』を常に点検する運用が重要です。


In [ ]:
pred = [0.10, 0.22, 0.31, 0.29]
real = [0.11, 0.25, 0.28, 0.35]
errors = [abs(p - r) for p, r in zip(pred, real)]
print('errors =', [round(e, 4) for e in errors])
print('mean_error =', round(sum(errors) / len(errors), 5))

平均誤差だけでなく時点別誤差を追うと、どの遷移条件でモデルが弱いかを特定しやすくなります。



## 要点整理

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 1ステップ誤差だけで安心してしまう
2. 長期予測の誤差爆発を監視しない
3. 状態表現が不足しているのにモデルだけ調整する
